In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/horse_health_outcomes/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Get a summary of the dataset
print(train_data.info())

# Check for missing values
print(train_data.isnull().sum())

# Visualize missing values
plt.figure(figsize=(10, 6))
sns.heatmap(train_data.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

# Distinguish column types
numeric_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object', 'category']).columns

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Visualize distribution of numeric columns
train_data[numeric_cols].hist(bins=20, figsize=(15, 10))
plt.suptitle('Distribution of Numeric Columns')
plt.show()

# Visualize distribution of categorical columns
for col in categorical_cols:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=train_data, x=col)
    plt.title(f'Distribution of {col}')
    plt.xticks(rotation=45)
    plt.show()

# Check for anomalies in numeric columns using boxplots
train_data[numeric_cols].boxplot(figsize=(15, 10))
plt.title('Boxplot of Numeric Columns')
plt.xticks(rotation=45)
plt.show()

# Correlation matrix for numeric columns
corr_matrix = train_data[numeric_cols].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Columns')
plt.show()


  surgery  hospital_number  ...  capillary_refill_time     outcome
0     yes           527706  ...             less_3_sec        died
1     yes           528641  ...             less_3_sec       lived
2     yes           535043  ...             more_3_sec  euthanized
3     yes           535043  ...             less_3_sec  euthanized
4     yes           528890  ...             more_3_sec        died

[5 rows x 9 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 986 entries, 0 to 985
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   surgery                986 non-null    object 
 1   hospital_number        986 non-null    int64  
 2   rectal_temp            986 non-null    float64
 3   pulse                  986 non-null    float64
 4   respiratory_rate       986 non-null    float64
 5   peripheral_pulse       938 non-null    object 
 6   mucous_membrane        971 non-null    object 
 7  

Numeric Columns: Index(['hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate'], dtype='object')
Categorical Columns: Index(['surgery', 'peripheral_pulse', 'mucous_membrane',
       'capillary_refill_time', 'outcome'],
      dtype='object')


In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-15 00:08:04.375 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['surgery', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time', 'outcome'], 'Numeric': ['hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Load the test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/horse_health_outcomes/test.csv')

# Handle missing values
fill_missing = FillMissingValue(features=['peripheral_pulse', 'mucous_membrane', 'capillary_refill_time'], strategy='most_frequent')
train_data = fill_missing.fit_transform(train_data)
test_data = fill_missing.transform(test_data)

# Encode categorical variables
label_encode = LabelEncode(features=['surgery', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time'])
train_data = label_encode.fit_transform(train_data)
test_data = label_encode.transform(test_data)

# Normalize numerical features
standard_scale = StandardScale(features=['hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate'])
train_data = standard_scale.fit_transform(train_data)
test_data = standard_scale.transform(test_data)

# Display the preprocessed data
print(train_data.head())
print(test_data.head())


   surgery  hospital_number  ...  capillary_refill_time     outcome
0        2        -0.322836  ...                      0        died
1        2        -0.322161  ...                      0       lived
2        2        -0.317536  ...                      1  euthanized
3        2        -0.317536  ...                      0  euthanized
4        2        -0.321981  ...                      1        died

[5 rows x 9 columns]
   surgery  hospital_number  ...  capillary_refill_time     outcome
0        0        -0.317292  ...                      0  euthanized
1        2        -0.317546  ...                      0  euthanized
2        2        -0.321568  ...                      1        died
3        2        -0.318176  ...                      0  euthanized
4        2        -0.321340  ...                      0       lived

[5 rows x 9 columns]


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


column_info
{'Category': ['outcome'], 'Numeric': ['surgery', 'hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from metagpt.tools.libs.data_preprocess import OneHotEncode

# Assuming train_data and test_data are already preprocessed from previous steps

# Separate features and target variable
X_train = train_data.drop('outcome', axis=1)
y_train = train_data['outcome']
X_test = test_data.drop('outcome', axis=1)
y_test = test_data['outcome']

# One-hot encode categorical columns
one_hot_encode = OneHotEncode(features=['surgery', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time'])
X_train = one_hot_encode.fit_transform(X_train)
X_test = one_hot_encode.transform(X_test)

# Initialize and train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate F1 score
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"F1 Score: {f1}")


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


F1 Score: 0.6330759024257476
